# PhoBERT Traditional Baseline - TF-IDF + SVM (Google Colab CPU)

Notebook nay su dung mo hinh truyen thong TF-IDF + SVM cho bai toan phan loai cam xuc tieng Viet.

Muc tieu:
- Huan luyen nhanh tren CPU cua Google Colab.
- Do chinh xac cao va on dinh tren du lieu cua ban.
- Luu model va metrics vao Google Drive.

Luu y quan trong:
- Scikit-learn SVM chu yeu chay CPU, khong can GPU.
- De nhanh toi da: dung LinearSVC + CalibratedClassifierCV + RandomizedSearchCV + n_jobs=-1.

In [ ]:
# Cai dat bo thu vien on dinh cho Google Colab Python 3.12
# Luu y: khong dung numpy 2.1.x de tranh loi ImportError voi scipy/sklearn
!pip -q install --upgrade --force-reinstall \
  "numpy==2.0.2" "scipy==1.13.1" "pandas==2.2.2" \
  "scikit-learn==1.6.1" "matplotlib==3.8.4" "seaborn==0.13.2" "joblib==1.4.2"

print('Da cai dat xong thu vien tuong thich.')


In [ ]:
import os
import re
import json
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from google.colab import drive
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import joblib

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

HAS_SEABORN = True
try:
    import seaborn as sns
except Exception as e:
    HAS_SEABORN = False
    sns = None
    print('Canh bao: khong import duoc seaborn. Se dung matplotlib de ve bieu do.')
    print('Chi tiet loi:', str(e)[:200])

LABEL_MAP = {0: 'Tiêu cực', 1: 'Tích cực', 2: 'Trung tính'}
print('Scikit-learn TF-IDF + SVM đã sẵn sàng (Google Colab CPU).')
print('HAS_SEABORN =', HAS_SEABORN)

In [ ]:
# Cau hinh Google Colab CPU + luu vao Google Drive (cung 1 tai khoan)
drive.mount('/content/drive')

BASE_DIR = '/content/drive/MyDrive/DO_AN_1_main'
DATA_FILE = f'{BASE_DIR}/data/data_v4_tu_crawl/data_cleaned_finetune.csv'
OUT_DIR = f'{BASE_DIR}/TF_IDF-SVM-v1'
os.makedirs(OUT_DIR, exist_ok=True)

if not os.path.exists(DATA_FILE):
    raise FileNotFoundError(f'Khong tim thay file du lieu: {DATA_FILE}')

# Giu ten bien DATA_PATH de tuong thich cac cell ben duoi
DATA_PATH = DATA_FILE

print('Dang chay tren Google Colab CPU.')
print('BASE_DIR:', BASE_DIR)
print('DATA_FILE ton tai:', os.path.exists(DATA_FILE))
print('OUT_DIR:', OUT_DIR)

In [ ]:
df = pd.read_csv(DATA_PATH)
df = df.rename(columns={'Review': 'text', 'Label_number': 'label', 'Label_text': 'label_text'})
df = df[['text', 'label', 'label_text']].copy()
df = df.dropna(subset=['text', 'label'])
df['text'] = df['text'].astype(str)
df['label'] = df['label'].astype(int)

def clean_text(s: str) -> str:
    s = s.lower().strip()
    s = re.sub(r'\s+', ' ', s)
    return s

df['text'] = df['text'].map(clean_text)
df = df[df['text'].str.len() > 2].reset_index(drop=True)

print(f'Tổng số mẫu: {len(df):,}')
print(df['label'].value_counts().sort_index())
display(df.head(5))

In [ ]:
plt.figure(figsize=(8, 5))
counts = df['label'].value_counts().sort_index()
labels_vi = [LABEL_MAP[i] for i in counts.index]

if HAS_SEABORN:
    sns.barplot(x=labels_vi, y=counts.values, palette='mako')
else:
    plt.bar(labels_vi, counts.values)

plt.title('Phân bố nhãn dữ liệu huấn luyện', fontsize=13, fontweight='bold')
plt.xlabel('Nhóm cảm xúc')
plt.ylabel('Số lượng')
for i, v in enumerate(counts.values):
    plt.text(i, v, f'{v:,}', ha='center', va='bottom', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
train_df, temp_df = train_test_split(
    df, test_size=0.2, random_state=SEED, stratify=df['label']
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=SEED, stratify=temp_df['label']
)

X_train, y_train = train_df['text'].values, train_df['label'].values
X_val, y_val = val_df['text'].values, val_df['label'].values
X_test, y_test = test_df['text'].values, test_df['label'].values

print(f'Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}')

class_weights_arr = compute_class_weight(class_weight='balanced', classes=np.array([0, 1, 2]), y=y_train)
class_weights = {i: float(w) for i, w in enumerate(class_weights_arr)}
print('Class weight:', {LABEL_MAP[i]: w for i, w in class_weights.items()})

In [ ]:
# Pipeline nhanh: TF-IDF (word+char n-gram) + LinearSVC + calibration xac suat
pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        lowercase=True,
        strip_accents=None,
        sublinear_tf=True,
        norm='l2',
    )),
    ('svm', LinearSVC(class_weight=class_weights, random_state=SEED, max_iter=3000))
])

param_distributions = {
    'tfidf__ngram_range': [(1, 1), (1, 2)],
    'tfidf__min_df': [2, 3, 5],
    'tfidf__max_df': [0.9, 0.95, 0.98],
    'tfidf__max_features': [60000, 90000, 120000],
    'tfidf__analyzer': ['word', 'char_wb'],
    'tfidf__token_pattern': [r'(?u)\b\w\w+\b'],
    'svm__C': [0.5, 1.0, 1.5, 2.0],
}

search = RandomizedSearchCV(
    estimator=pipeline,
    param_distributions=param_distributions,
    n_iter=12,
    scoring='f1_macro',
    n_jobs=-1,
    cv=3,
    verbose=1,
    random_state=SEED,
)

search.fit(X_train, y_train)
print('Best CV F1:', round(search.best_score_, 4))
print('Best params:', search.best_params_)

In [ ]:
# Hieu chinh xac suat de co do tu tin tot hon
best_pipe = search.best_estimator_
calibrated_clf = CalibratedClassifierCV(best_pipe, method='sigmoid', cv=3)
calibrated_clf.fit(X_train, y_train)

val_pred = calibrated_clf.predict(X_val)
val_acc = accuracy_score(y_val, val_pred)
val_p, val_r, val_f1, _ = precision_recall_fscore_support(y_val, val_pred, average='macro', zero_division=0)
print(f'Val Acc: {val_acc*100:.2f}% | Val F1 macro: {val_f1*100:.2f}%')

In [ ]:
# Danh gia tren tap test
y_pred = calibrated_clf.predict(X_test)
y_prob = calibrated_clf.predict_proba(X_test)

acc = accuracy_score(y_test, y_pred)
p, r, f1, _ = precision_recall_fscore_support(y_test, y_pred, average='macro', zero_division=0)

print(f'Độ chính xác Test: {acc*100:.2f}%')
print(f'Precision macro: {p*100:.2f}%')
print(f'Recall macro: {r*100:.2f}%')
print(f'F1 macro: {f1*100:.2f}%')

print('\nBáo cáo phân loại:')
print(classification_report(y_test, y_pred, target_names=[LABEL_MAP[0], LABEL_MAP[1], LABEL_MAP[2]], digits=4))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(8, 6))
if HAS_SEABORN:
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='YlGnBu',
        xticklabels=[LABEL_MAP[0], LABEL_MAP[1], LABEL_MAP[2]],
        yticklabels=[LABEL_MAP[0], LABEL_MAP[1], LABEL_MAP[2]],
    )
else:
    plt.imshow(cm, cmap='YlGnBu')
    plt.colorbar()
    ticks = [0, 1, 2]
    labels = [LABEL_MAP[0], LABEL_MAP[1], LABEL_MAP[2]]
    plt.xticks(ticks, labels)
    plt.yticks(ticks, labels)
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            plt.text(j, i, str(cm[i, j]), ha='center', va='center')

plt.title('Ma trận nhầm lẫn - TF-IDF + SVM', fontsize=13, fontweight='bold')
plt.xlabel('Nhãn dự đoán')
plt.ylabel('Nhãn thực tế')
plt.tight_layout()
plt.show()

# Phan bo do tu tin cua du doan dung/sai
conf = y_prob.max(axis=1)
correct = (y_pred == y_test)

plt.figure(figsize=(8, 5))
if HAS_SEABORN:
    sns.kdeplot(conf[correct], fill=True, label='Dự đoán đúng')
    sns.kdeplot(conf[~correct], fill=True, label='Dự đoán sai')
else:
    plt.hist(conf[correct], bins=30, alpha=0.6, density=True, label='Dự đoán đúng')
    plt.hist(conf[~correct], bins=30, alpha=0.6, density=True, label='Dự đoán sai')

plt.title('Phân bố độ tự tin của mô hình', fontsize=13, fontweight='bold')
plt.xlabel('Độ tự tin (xác suất lớn nhất)')
plt.ylabel('Mật độ')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
def predict_sentiment(text: str):
    text = re.sub(r'\s+', ' ', str(text).lower().strip())
    pred = int(calibrated_clf.predict([text])[0])
    prob = calibrated_clf.predict_proba([text])[0]
    conf = float(np.max(prob))
    return LABEL_MAP[pred], conf, prob

samples = [
    'sản phẩm quá tệ, mới dùng đã hỏng',
    'giao hàng nhanh, đóng gói ổn',
    'chất lượng tuyệt vời, sẽ ủng hộ tiếp',
    '5 sao cho vui chứ không bao giờ mua lại',
]

for s in samples:
    label, conf, prob = predict_sentiment(s)
    print('-' * 90)
    print('Câu:', s)
    print('Dự đoán:', label, f'| Độ tự tin: {conf*100:.2f}%')
    print('Xác suất [Tiêu cực, Tích cực, Trung tính]:', np.round(prob, 4))

In [ ]:
joblib.dump(calibrated_clf, f'{OUT_DIR}/tfidf_svm_calibrated.joblib')

metrics = {
    'model': 'TF-IDF + LinearSVC + CalibratedClassifierCV',
    'test_accuracy': float(acc),
    'test_precision_macro': float(p),
    'test_recall_macro': float(r),
    'test_f1_macro': float(f1),
    'best_cv_f1': float(search.best_score_),
    'best_params': search.best_params_,
    'train_samples': int(len(X_train)),
    'val_samples': int(len(X_val)),
    'test_samples': int(len(X_test)),
}

with open(f'{OUT_DIR}/metrics_tfidf_svm.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, ensure_ascii=False, indent=2)

print('Đã lưu model và metrics vào:', OUT_DIR)
print(json.dumps(metrics, ensure_ascii=False, indent=2))